# AutoEncoders

## What is an AutoEncoder?

An **autoencoder (AE)** is a neural network that learns to reconstruct its input. Instead of requiring separate labels, it uses the input itself as the training target, learning useful features through a constrained representation.

### Encoder, latent representation, and decoder

- **Encoder:** maps the input $\mathbf{x}$ to a latent representation $\mathbf{z}$.
- **Bottleneck:** restricts the information available to the decoder, encouraging the model to capture important patterns rather than simply copy the input.
- **Decoder:** maps $\mathbf{z}$ back to a reconstruction $\hat{\mathbf{x}}$ with the same dimensions as the input.

$$
\mathbf{z} = f_{\theta}(\mathbf{x}), \qquad \hat{\mathbf{x}} = g_{\phi}(\mathbf{z})
$$

Here, $\theta$ and $\phi$ are the trainable encoder and decoder parameters. In an undercomplete autoencoder, the latent representation has fewer dimensions than the input. A narrow bottleneck is one way to constrain the model; sparsity, noise, and other regularization methods are alternatives.

```mermaid
flowchart LR
    X["Input x: movie ratings"] --> E["Encoder: 20 units"]
    E --> Z["Bottleneck z: 10 units"]
    Z --> D["Decoder: 20 units"]
    D --> R["Reconstruction: predicted ratings"]
```

### How does it learn?

Training minimizes a **reconstruction loss** using backpropagation. For continuous inputs, a common choice is mean squared error (MSE):

$$
\mathcal{L}(\theta,\phi) = \frac{1}{Nd}\sum_{i=1}^{N}\sum_{j=1}^{d}\left(x_{ij}-\hat{x}_{ij}\right)^2
$$

Here, $N$ is the number of examples and $d$ is the number of input features. A lower reconstruction loss means the outputs more closely match the inputs, but does not by itself guarantee useful latent features or good generalization.

### In this notebook: movie recommendations

Each user is represented by a vector with one entry per movie. Observed entries contain ratings; missing ratings are stored as zero, which is a placeholder rather than a negative preference. The **stacked autoencoder (SAE)** uses multiple layers with widths $d \rightarrow 20 \rightarrow 10 \rightarrow 20 \rightarrow d$, where $d$ is the number of movies. Hidden layers use sigmoid activations, while the output layer is linear to predict continuous ratings.

Only observed ratings should contribute to reconstruction error. With $\Omega$ denoting the set of observed user–movie pairs, the observed-rating MSE is:

$$
\mathcal{L}_{\mathrm{observed}} = \frac{1}{|\Omega|}\sum_{(u,m)\in\Omega}\left(r_{um}-\hat{r}_{um}\right)^2
$$

The training code masks missing entries before computing MSE and rescales the reported error for the number of rated movies. After training, predictions for previously unrated movies can be ranked to suggest recommendations. Evaluate on held-out ratings, not merely on the ratings used for training.

### Common variants and uses

- **Undercomplete autoencoders:** learn compact representations for dimensionality reduction and feature extraction.
- **Denoising autoencoders:** reconstruct clean inputs from corrupted inputs to learn robust features.
- **Sparse autoencoders:** penalize latent activations to encourage selective feature use.
- **Variational autoencoders (VAEs):** learn a probabilistic latent representation with an additional regularization objective, enabling generation of new samples.

Autoencoders also support anomaly detection: unusually high reconstruction error can indicate an unfamiliar input, although this requires validation and is not guaranteed. Without suitable constraints, an expressive model may simply memorize its training data.


## Importing the libraries

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.utils.data

## Importing the dataset

In [ ]:
movies = pd.read_csv('ml-1m/movies.dat', sep = '::', header = None, engine = 'python', encoding = 'latin-1')
users = pd.read_csv('ml-1m/users.dat', sep = '::', header = None, engine = 'python', encoding = 'latin-1')
ratings = pd.read_csv('ml-1m/ratings.dat', sep = '::', header = None, engine = 'python', encoding = 'latin-1')

## Preparing the training set and the test set

In [ ]:
training_set = pd.read_csv('ml-100k/u1.base', delimiter = '\t')
training_set = np.array(training_set, dtype = 'int')
test_set = pd.read_csv('ml-100k/u1.test', delimiter = '\t')
test_set = np.array(test_set, dtype = 'int')

## Getting the number of users and movies

In [ ]:
nb_users = int(max(max(training_set[:,0]), max(test_set[:,0])))
nb_movies = int(max(max(training_set[:,1]), max(test_set[:,1])))

## Converting the data into an array with users in lines and movies in columns

In [ ]:
def convert(data):
    new_data = []
    for id_users in range(1, nb_users + 1):
        id_movies = data[:,1][data[:,0] == id_users]
        id_ratings = data[:,2][data[:,0] == id_users]
        ratings = np.zeros(nb_movies)
        ratings[id_movies - 1] = id_ratings
        new_data.append(list(ratings))
    return new_data
training_set = convert(training_set)
test_set = convert(test_set)

## Converting the data into Torch tensors

In [ ]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'PyTorch device: {device}')
training_set = torch.as_tensor(training_set, dtype=torch.float32, device=device)
test_set = torch.as_tensor(test_set, dtype=torch.float32, device=device)

## Creating the architecture of the Neural Network

In [ ]:
class SAE(nn.Module):
    def __init__(self, ):
        super(SAE, self).__init__()
        self.fc1 = nn.Linear(nb_movies, 20)
        self.fc2 = nn.Linear(20, 10)
        self.fc3 = nn.Linear(10, 20)
        self.fc4 = nn.Linear(20, nb_movies)
        self.activation = nn.Sigmoid()
    def forward(self, x):
        x = self.activation(self.fc1(x))
        x = self.activation(self.fc2(x))
        x = self.activation(self.fc3(x))
        x = self.fc4(x)
        return x
sae = SAE().to(device)
criterion = nn.MSELoss()
optimizer = optim.RMSprop(sae.parameters(), lr = 0.01, weight_decay = 0.5)

## Training the SAE

In [ ]:
nb_epoch = 200
sae.train()
for epoch in range(1, nb_epoch + 1):
    train_loss = 0
    s = 0.
    for id_user in range(nb_users):
        inputs = training_set[id_user].unsqueeze(0)
        rated = inputs > 0
        rated_count = rated.sum()
        if rated_count > 0:
            optimizer.zero_grad()
            output = sae(inputs).masked_fill(~rated, 0)
            loss = criterion(output, inputs)
            loss.backward()
            optimizer.step()
            train_loss += (loss.detach() * nb_movies / rated_count).sqrt()
            s += 1.
    print(f'epoch: {epoch} loss: {(train_loss / s).item()}')

## Testing the SAE

In [ ]:
test_loss = 0
s = 0.
sae.eval()
with torch.no_grad():
    for id_user in range(nb_users):
        inputs = training_set[id_user].unsqueeze(0)
        target = test_set[id_user].unsqueeze(0)
        rated = target > 0
        rated_count = rated.sum()
        if rated_count > 0:
            output = sae(inputs).masked_fill(~rated, 0)
            loss = criterion(output, target)
            test_loss += (loss * nb_movies / rated_count).sqrt()
            s += 1.
print(f'test loss: {(test_loss / s).item()}')